# Visualization of loops in contact maps in *M. flavus*

## Import libraries

In [ ]:
# import standard python libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib as mpl

In [ ]:
# import cooltools and cooler
import cooltools
import cooler

In [ ]:
# Import libraries for plotting
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cooltools.lib.plotting

from matplotlib.ticker import EngFormatter # For adding bp to plots
import matplotlib.patches as patches # For adding circles to the plots

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

## Check available resolutions in mcool file and import the file at 5 kb resolution

In [ ]:
# Resolutions in mcool file
cooler.fileops.list_coolers('/Users/emma/Documents/NMBU/Master/Master/gzMucFlav1/Subgenome2/gzMucFlav1_sub2.mcool')

In [ ]:
# Load the cool file at a specific resolution
res_5k = cooler.Cooler('/Users/emma/Documents/NMBU/Master/Master/gzMucFlav1/Subgenome2/gzMucFlav1_sub2.mcool::resolutions/5000') 

In [ ]:
# Makes a list of chromosomes and the bins for each chromosome. 
chromstarts = []
for i in res_5k.chromnames:
    print(f'{i} : {res_5k.extent(i)}')
    chromstarts.append(res_5k.extent(i)[0])

# Filtering out scaffolds from the Hi-C data

In [ ]:
# Keep only chromosomes/scaffolds that are large enough
chroms_to_keep_5k = [c for c in res_5k.chromnames if "Scaffold" not in c and "MT" not in c]

print("Chromosomes kept:", chroms_to_keep_5k) # Print the chromosomes that we plot. 

In [ ]:
# Find the bins for the chromosomes we are going to visualize and get the matrix for these chromosomes.

## Get the bins for the chromosomes we keep
bins_res_5k = res_5k.bins()[:]
keep_mask_res5k = bins_res_5k['chrom'].isin(chroms_to_keep_5k)

In [ ]:
# Fetch full genome-wide matrix
full_matrix_res5k = res_5k.matrix(balance=False)[:]

# Subset matrix to only keep large chromosomes
matrix_filtered_res5k = full_matrix_res5k[keep_mask_res5k.values, :][:, keep_mask_res5k.values]

In [ ]:
# Shorten the chromosome names so they only show the number of the chromosome.
chromstarts = []
pos = 0
short_labels = []
for c in chroms_to_keep_5k:
    n_bins = (bins_res_5k[bins_res_5k['chrom'] == c].shape[0])
    chromstarts.append(pos)
    short_labels.append(c.split('_')[-1])
    pos += n_bins

print(short_labels)

Used formatting code from cooltools tutorial: [https://cooltools.readthedocs.io/en/latest/notebooks/viz.html](https://cooltools.readthedocs.io/en/latest/notebooks/viz.html)

In [ ]:
# Plots ticks as megabases
bp_formatter = EngFormatter('b')

def format_ticks(ax, x=True, y=True, rotate=True):
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
        ax.xaxis.tick_top()  # move x ticks to top for genome plots
    if rotate:
        ax.tick_params(axis='x')

# Import loop file generated by Mustache

Loops were called at 5 kb resolution using Mustache with the parameters, p-threshold of 0.2 and sparsity-threshold of 0.7.

## Importing and processing the loop file

In [ ]:
loops_raw = pd.read_csv(
    "/Users/emma/mustache/5kb_p02_st07.tsv",
    sep="\t",
    header=0  # Has header
)
# Keep only the columns we care about
# BIN1_CHR, BIN1_START, BIN1_END, BIN2_CHROMOSOME, BIN2_START
loops = loops_raw[["BIN1_CHR", "BIN1_START", "BIN1_END", "BIN2_CHROMOSOME", "BIN2_START", "BIN2_END", "FDR"]].copy()

# Remove 'subgenome2_' prefix if present in any chromosome column
loops["BIN1_CHR"] = loops["BIN1_CHR"].str.replace("^subgenome2_", "", regex=True)
loops["BIN2_CHROMOSOME"] = loops["BIN2_CHROMOSOME"].str.replace("^subgenome2_", "", regex=True)

# Rename columns to standard BEDPE format
loops_5k_bedpe = loops.rename(columns={
    "BIN1_CHR": "chrom1",
    "BIN1_START": "start1",
    "BIN1_END": "end1",
    "BIN2_CHROMOSOME": "chrom2",
    "BIN2_START": "start2",
    "BIN2_END": "end2"
})

print(loops_5k_bedpe.head())
print(loops_5k_bedpe.shape) # To see the size, 17 rows and 7 columns.

## Find size and mean size of loops

In [ ]:
loops_5k_bedpe['loop_size'] = loops_5k_bedpe['start2'] - loops_5k_bedpe['start1']
# Convert loop size to Kb
loops_5k_bedpe['loop_size_kb'] = loops_5k_bedpe['loop_size'] / 1_000

# Check the first few
print(loops_5k_bedpe[['chrom1','start1','start2','loop_size_kb']].head())

# The total number of called loops
total_loops = len(loops_5k_bedpe)
print("Total number of called loops:", total_loops)

# Smallest loop
min_loop = loops_5k_bedpe.loc[loops_5k_bedpe['loop_size_kb'].idxmin()]
print("Smallest loop:")
print(min_loop[['chrom1','start1','start2','loop_size_kb']])

# Largest loop
max_tad = loops_5k_bedpe.loc[loops_5k_bedpe['loop_size_kb'].idxmax()]
print("\nLargest loop:")
print(max_tad[['chrom1','start1','start2','loop_size_kb']])

# Calculate mean loop size in kb
mean_loop_size_kb = loops_5k_bedpe['loop_size_kb'].mean()
print("Mean loop size (kb):", mean_loop_size_kb)

## Loop through all chromatin loops called by Mustache

In [ ]:
binsize = 5000  # matrix resolution

# For loop which goes through all the loops in the file and plots them 
for idx, loop in loops_5k_bedpe.iterrows():
    chrom = loop["chrom1"]
    start1 = loop["start1"]
    end1 = loop["end1"]
    start2 = loop["start2"]
    
    # Get the short chromosome label 
    chrom_index = chroms_to_keep_5k.index(chrom)
    chrom_label = short_labels[chrom_index]
    
    # Define region around loop
    pad = 200_000  # 200 kb upstream/downstream

    # Get chromosome size from cooler
    chrom_length = res_5k.chromsizes[chrom]

    # Defines the region
    # Start position is the smallest position of start1 and start2, and adds 200 kb upstream
    region_start = max(0, min(start1, start2) - pad)
    # End position is the largest position of end1 and start 2, and adds 200 kb downstream. 
    # Chrom_length ensures that the end coordinate is not further than the chromosome length
    region_end = min(chrom_length, max(end1, start2) + pad) 
    # String of the region
    region_str = f"{chrom}:{region_start}-{region_end}"
    extents = (region_start, region_end, region_end, region_start)
    
    # Fetch matrix (for the given region) from cooler 
    matrix = res_5k.matrix(balance=False).fetch(region_str)
    
    # Plotting
    fig, ax = plt.subplots(figsize=(6,6))
    # Logarithmic scaling of the matrix
    norm = LogNorm(vmin=1, vmax=matrix.max())
    # Visualize the contact matrix with fall color map. x and y range match region defined above. 
    im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)
    # Add colorbar indicating contact frequency
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Contact counts (log scale)')
    
    # Marking the loop center
    ax.scatter(start2, start1, color='black', s=30, alpha = 0.3, label='Loop center')
    ax.legend()
    
    # Axes labels and title
    ax.set_xlabel('Genomic position')
    ax.set_ylabel('Genomic position')
    ax.set_title(f"Loop {idx} on Chromosome {chrom_label}")

    # Use the tick formatting function from cooltools tutorial
    format_ticks(ax)

    plt.tight_layout()
    plt.show()

    # Print the FDR value and loop size for the given loop.
    fdr_value = loop["FDR"] 
    loop_size = loop["loop_size_kb"]
    print(f"FDR: {fdr_value}") 
    print(f"Loop size in Kb: {loop_size}")
    print("----")

# Adds annotation of neighbourhood

In [ ]:
binsize = 5000   # Matrix resolution
n = 2            # Neighborhood size (1->3x3, 2->5x5)

for idx, loop in loops_5k_bedpe.iterrows():

    chrom = loop["chrom1"]
    start1 = loop["start1"]
    end1 = loop["end1"]
    start2 = loop["start2"]

    # Chromosome label
    chrom_index = chroms_to_keep_5k.index(chrom)
    chrom_label = short_labels[chrom_index]

    # Region around loop
    pad = 200_000
    chrom_length = res_5k.chromsizes[chrom]

    region_start = max(0, min(start1, start2) - pad)
    region_end = min(chrom_length, max(end1, start2) + pad)

    region_str = f"{chrom}:{region_start}-{region_end}"

    # Fetch Hi-C matrix
    matrix = res_5k.matrix(balance=False).fetch(region_str)

    if matrix.size == 0:
        print(f"Loop {idx} skipped (empty matrix)")
        continue

    rows, cols = matrix.shape

    # Compute matrix bin indices
    i = int((start1 - region_start) // binsize)
    j = int((start2 - region_start) // binsize)

    # Skip loops outside matrix
    if i >= rows or j >= cols:
        print(f"Loop {idx} skipped (loop outside fetched matrix)")
        continue

    # Neighborhood bounds
    i_min = max(i - n, 0)
    i_max = min(i + n + 1, rows)

    j_min = max(j - n, 0)
    j_max = min(j + n + 1, cols)

    # Ensure full neighborhood exists
    if (i_max - i_min) < (2*n + 1) or (j_max - j_min) < (2*n + 1):
        print(f"Loop {idx} skipped (too close to edge)")
        continue

    neighborhood = matrix[i_min:i_max, j_min:j_max]

    if neighborhood.size == 0:
        print(f"Loop {idx} skipped (empty neighborhood)")
        continue

    # Center pixel
    center = matrix[i, j]

    # Background
    neighbors = np.delete(neighborhood.flatten(), n*(2*n+1) + n)
    avg_bg = neighbors.mean()

    # Rectangle genomic coordinates
    x0 = region_start + j_min * binsize
    y0 = region_start + i_min * binsize
    width = (j_max - j_min) * binsize
    height = (i_max - i_min) * binsize

    # Plotting
    fig, ax = plt.subplots(figsize=(6,6))

    norm = LogNorm(vmin=1, vmax=np.nanmax(matrix))

    extents = (region_start, region_end, region_end, region_start)

    im = ax.matshow(
        matrix,
        cmap="fall",
        norm=norm,
        extent=extents
    )

    plt.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04,
        label="Contact counts (log scale)"
    )

    # Loop center
    ax.scatter(
        start2,
        start1,
        color="black",
        s=30,
        alpha=0.4,
        label="Loop center"
    )

    # Neighborhood square
    rect = patches.Rectangle(
        (x0, y0),
        width,
        height,
        linewidth=2,
        edgecolor="blue",
        facecolor="none",
        alpha=0.7
    )

    ax.add_patch(rect)

    ax.legend()

    ax.set_xlabel("Genomic position")
    ax.set_ylabel("Genomic position")

    ax.set_title(f"Loop {idx} on Chromosome {chrom_label} in M. flavus")

    format_ticks(ax)

    plt.tight_layout()
    plt.show()

    # Enrichment results
    print(f"Loop {idx} contact count: {center}")
    print(f"Average background ({2*n+1}x{2*n+1}): {avg_bg:.2f}")

    if center > avg_bg:
        print("Loop appears enriched")
    else:
        print("Loop does NOT appear enriched")

    # Print the FDR value and loop size for the given loop.
    fdr_value = loop["FDR"] 
    loop_size = loop["loop_size_kb"]
    print(f"FDR: {fdr_value}") 
    print(f"Loop size in Kb: {loop_size}")
    print("----")

## Loops used in results

In [ ]:
# Set fontsizes for result plots (to be the same across species)
mpl.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14
})

In [ ]:
# Pick a loop
loop = loops_5k_bedpe.iloc[16]

chrom = loop["chrom1"]
start1 = loop["start1"]
end1 = loop["end1"]
start2 = loop["start2"]


# Finds the chromosome name in chroms_to_keep and then finds the chromosome in short_labels, finding the chromosome number
chrom_index = chroms_to_keep_5k.index(chrom)  
chrom_label = short_labels[chrom_index]  

chrom_index = chroms_to_keep_5k.index(chrom)
chrom_label = short_labels[chrom_index]
    
pad = 200_000  # 200 kb upstream/downstream
    # Get chromosome size from cooler
chrom_length = res_5k.chromsizes[chrom]
region_start = max(0, min(start1, start2) - pad)
region_end = min(chrom_length, max(end1, start2) + pad)
region_str = f"{chrom}:{region_start}-{region_end}"
extents = (region_start, region_end, region_end, region_start)

# Fetch the matrix for this region
# Assuming you have a cooler object called res_5k
matrix = res_5k.matrix(balance=False).fetch(region_str)

# Plot
f, ax = plt.subplots(figsize=(6, 6))
norm = LogNorm(vmin=1, vmax=matrix.max())

im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Contact counts (log scale)')
cbar.ax.tick_params()

# Add loop dot
ax.scatter(start2, start1, color='black', s=30, alpha=0.3)

# Add FDR as text on the plot (bottom-left corner)
fdr_value = loop["FDR"]
ax.text(
    0.01, 0.01, f"FDR = {fdr_value:.4f}", 
    transform=ax.transAxes,  # use axis coordinates (0-1)
    fontsize=14, 
    verticalalignment='bottom')

# Axes labels
ax.set_xlabel('Genomic Position')
ax.set_ylabel('Genomic Position')

# Tick formatting function
format_ticks(ax)

ax.set_title(f"Chromosome {chrom_label} in M. flavus", fontsize=18, color='#E57373')

plt.tight_layout()
plt.show()

In [ ]:
# Pick a loop
loop = loops_5k_bedpe.iloc[16]

chrom = loop["chrom1"]
start1 = loop["start1"]
end1 = loop["end1"]
start2 = loop["start2"]


# Finds the chromosome name in chroms_to_keep and then finds the chromosome in short_labels, finding the chromosome number
chrom_index = chroms_to_keep_5k.index(chrom)  
chrom_label = short_labels[chrom_index]  

chrom_index = chroms_to_keep_5k.index(chrom)
chrom_label = short_labels[chrom_index]
    
pad = 100_000  # 100 kb upstream/downstream
    # Get chromosome size from cooler
chrom_length = res_5k.chromsizes[chrom]
region_start = max(0, min(start1, start2) - pad)
region_end = min(chrom_length, max(end1, start2) + pad)
region_str = f"{chrom}:{region_start}-{region_end}"
extents = (region_start, region_end, region_end, region_start)

# Fetch the matrix for this region
# Assuming you have a cooler object called res_5k
matrix = res_5k.matrix(balance=False).fetch(region_str)

# Plot
f, ax = plt.subplots(figsize=(6, 6))
norm = LogNorm(vmin=1, vmax=matrix.max())

im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Contact counts (log scale)', fontsize=12)
cbar.ax.tick_params(labelsize=11)

# Add loop dot
ax.scatter(start2, start1, color='black', s=30, alpha=0.3)

# Add FDR as text on the plot (bottom-left corner)
fdr_value = loop["FDR"]
ax.text(
    0.01, 0.01, f"FDR = {fdr_value:.4f}", 
    transform=ax.transAxes,  # use axis coordinates (0-1)
    fontsize=12, 
    verticalalignment='bottom')

# Axes labels
ax.set_xlabel('Genomic Position', fontsize=12)
ax.set_ylabel('Genomic Position', fontsize=12)

# Tick formatting function
format_ticks(ax)

ax.set_title(f"Chromosome {chrom_label} in M. flavus", fontsize=15)

plt.tight_layout()
plt.show()

In [ ]:
# Pick a loop
loop = loops_5k_bedpe.iloc[8]

chrom = loop["chrom1"]
start1 = loop["start1"]
end1 = loop["end1"]
start2 = loop["start2"]


# Finds the chromosome name in chroms_to_keep and then finds the chromosome in short_labels, finding the chromosome number
chrom_index = chroms_to_keep_5k.index(chrom)  
chrom_label = short_labels[chrom_index]  

chrom_index = chroms_to_keep_5k.index(chrom)
chrom_label = short_labels[chrom_index]
    
pad = 100_000  # 200 kb upstream/downstream
    # Get chromosome size from cooler
chrom_length = res_5k.chromsizes[chrom]
region_start = max(0, min(start1, start2) - pad)
region_end = min(chrom_length, max(end1, start2) + pad)
region_str = f"{chrom}:{region_start}-{region_end}"
extents = (region_start, region_end, region_end, region_start)

# Fetch the matrix for this region
# Assuming you have a cooler object called res_5k
matrix = res_5k.matrix(balance=False).fetch(region_str)

# Plot
f, ax = plt.subplots(figsize=(6, 6))
norm = LogNorm(vmin=1, vmax=matrix.max())

im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Contact counts (log scale)', fontsize=12)
cbar.ax.tick_params(labelsize=11)

# Add loop dot
ax.scatter(start2, start1, color='black', s=30, alpha=0.3)

# Add FDR as text on the plot (bottom-left corner)
fdr_value = loop["FDR"]
ax.text(
    0.01, 0.01, f"FDR = {fdr_value:.4f}", 
    transform=ax.transAxes,  # use axis coordinates (0-1)
    fontsize=12, 
    verticalalignment='bottom')

# Axes labels
ax.set_xlabel('Genomic Position', fontsize=12)
ax.set_ylabel('Genomic Position', fontsize=12)

# Tick formatting function
format_ticks(ax)

ax.set_title(f"Chromosome {chrom_label} in M. flavus", fontsize=15)

plt.tight_layout()
plt.show()